In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def clean_and_engineer_data(input_file, output_file):
    df = pd.read_csv(input_file)
    cols_to_drop = ['ID', 'Payment Plan']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    df['Home Ownership'] = pd.to_numeric(df['Home Ownership'], errors='coerce')
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    df[categorical_cols] = df[categorical_cols].fillna('Unknown')

    df['Loan_to_Income_Ratio'] = df['Loan Amount'] / (df['Home Ownership'].replace(0, np.nan))
    df['Repayment_Progress'] = df['Total Received Interest'] / (df['Funded Amount'].replace(0, np.nan))

    df[['Loan_to_Income_Ratio', 'Repayment_Progress']] = df[['Loan_to_Income_Ratio', 'Repayment_Progress']].fillna(0)

    le = LabelEncoder()
    for col in categorical_cols:
        num_unique = df[col].nunique()
        if num_unique > 5:
            df[col] = le.fit_transform(df[col].astype(str))
        else:
            dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
            df = pd.concat([df, dummies], axis=1)
            df.drop(columns=[col], inplace=True)

    df.to_csv(output_file, index=False)
    print(f"Success! Cleaned file saved as '{output_file}'")
    print(f"Original columns: 35 | New columns: {df.shape[1]}")

clean_and_engineer_data('train.csv', 'cleaned_loan_data.csv')

In [ ]:
!pip install -q sdv lightgbm imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.8/200.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00


**CatBoost**

In [ ]:
# Install necessary libraries
!pip install catboost imbalanced-learn shap dice-ml

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from catboost import CatBoostClassifier
from imblearn.over_sampling import ADASYN
import shap

# Load Dataset - Ensure 'cleaned_loan_data.csv.xls' is uploaded to Colab
df = pd.read_csv('cleaned_loan_data.csv.xls')

# Step 1: Prepare data
X = df.drop(columns=['Loan Status'])
y = df['Loan Status']

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Impute missing values using median
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Step 4: Yeo-Johnson Transformation
numeric_cols = X_train_imputed.select_dtypes(include=[np.number]).columns
yeo = PowerTransformer(method='yeo-johnson', standardize=False)
X_train_imputed[numeric_cols] = yeo.fit_transform(X_train_imputed[numeric_cols])
X_test_imputed[numeric_cols] = yeo.transform(X_test_imputed[numeric_cols])

# Step 5: Min-Max Scaling
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test.columns)

# Step 6: ADASYN Adaptive Resampling
# Focusing on "hard" examples to improve recall
def data_sampling(method = "SMOTE-Tomek"):
  if method == "SMOTE-Tomek":
    smt = SMOTETomek(
        sampling_strategy=0.4,
        tomek=TomekLinks(sampling_strategy='all'),
        random_state=42
    )
    X_res, y_res = smt.fit_resample(X_train_scaled, y_train)
    return X_res, y_res

  if method == "ADASYN":
    ada = ADASYN(sampling_strategy='auto', random_state=42)
    X_res, y_res = ada.fit_resample(X_train_scaled, y_train)
    return X_res, y_res

  return X_train_scaled, y_train  # No Sampling

X_res, y_res = data_sampling(
    method = "ADASYN"  # SMOTE-Tomek -- other method
)

# Step 7: Train CatBoost for SHAP Feature Selection
# Optimized for T4 GPU
cat_selector = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    task_type='GPU',
    verbose=0,
    random_seed=42
)
cat_selector.fit(X_res, y_res)

# Step 8: SHAP feature importance
explainer = shap.Explainer(cat_selector)
shap_values = explainer(X_res)
shap_importance = np.abs(shap_values.values).mean(axis=0)

shap_feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': shap_importance
}).sort_values(by='importance', ascending=False)

# Step 9: Subset to top 30 SHAP features
top_features = shap_feature_importance['feature'].head(30).tolist()
X_train_shap = X_res[top_features]
X_test_shap = X_test_scaled[top_features]

# Step 10: Optimized GridSearchCV on CatBoost
# Tuning for ROC-AUC to maximize class separation
params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1],
    'iterations': [100, 200],
    'scale_pos_weight': [5, 10, 20]
}

grid_cat = GridSearchCV(
    CatBoostClassifier(task_type='GPU', verbose=0, random_seed=42),
    param_grid=params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

grid_cat.fit(X_train_shap, y_res)
final_model = grid_cat.best_estimator_
y_pred_cat = final_model.predict(X_test_shap)

# Final Result Summary
print("🔷 Final Optimized CatBoost Results")
print("Best Parameters:", grid_cat.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_cat))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_cat))
print("Classification Report:\n", classification_report(y_test, y_pred_cat))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.2 MB/s eta 0:00:00
🔷 Final Optimized CatBoost Results
Best Parameters: {'depth': 8, 'iterations': 200, 'learning_rate': 0.1, 'scale_pos_weight': 20}
Accuracy: 0.2821462980804862
Confusion Matrix:
 [[2861 9384]
 [ 302  946]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.23      0.37     12245
           1       0.09      0.76      0.16      1248

    accuracy                           0.28     13493
   macro avg       0.50      0.50      0.27     13493
weighted avg       0.83      0.28      0.35     13493



**LightGBM**

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve

lgbm = LGBMClassifier(
    objective='binary',
    class_weight='balanced',      # Focus on minority
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_child_weight=10,
    random_state=42
)

lgbm.fit(X_train_shap, y_res)

# Predict and tune threshold
y_probs = lgbm.predict_proba(X_test_shap)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred = (y_probs >= best_thresh).astype(int)

print("\n🔷 Balanced LightGBM Results")
print(f"Best Threshold: {best_thresh:.2f}, F1: {max(f1_scores):.4f}")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 48976, number of negative: 48976
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030383 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6950
[LightGBM] [Info] Number of data points in the train set: 97952, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

🔷 Balanced LightGBM Results
Best Threshold: 0.04, F1: 0.1721
Accuracy: 0.17223745645890462
Confusion Matrix:
 [[ 1163 11082]
 [   87  1161]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.09      0.17     12245
           1       0.09      0.93      0.17      1248

    accuracy                           0.17     13493
   macro avg       0.51      0.51      0.17     13493
weighted avg       0.85      0.17      0.17

**XGBoost Unoptimized**

In [ ]:
# Step 10:
# Higher scale_pos_weight helps the model focus on the minority class (defaults)
params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1],
    # Adjusted to [10, 20, 50] to better match the ~10:1 class imbalance
    'scale_pos_weight': [10, 20, 50]
}

grid = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid=params,
    # Changed from 'f1' to 'roc_auc' to optimize class separation across thresholds
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_shap, y_res)
final_model = grid.best_estimator_
y_pred_xgb = final_model.predict(X_test_shap)

# Output results
print("🔷 Updated XGBoost GridSearch Results")
print("Best Parameters:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:37:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🔷 Updated XGBoost GridSearch Results
Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'scale_pos_weight': 20, 'subsample': 0.8}
Accuracy: 0.4696509301119099
Confusion Matrix:
 [[5666 6579]
 [ 577  671]]
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.46      0.61     12245
           1       0.09      0.54      0.16      1248

    accuracy                           0.47     13493
   macro avg       0.50      0.50      0.39     13493
weighted avg       0.83      0.47      0.57     13493



**XGBoost Optimized**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import TomekLinks
import shap

df = pd.read_csv('cleaned_loan_data.csv')

# Step 1: Prepare data
X = df.drop(columns=['Loan Status'])
y = df['Loan Status']

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Impute missing values using median
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Step 4: Yeo-Johnson Transformation (only numeric columns)
numeric_cols = X_train_imputed.select_dtypes(include=[np.number]).columns
yeo = PowerTransformer(method='yeo-johnson', standardize=False)
X_train_imputed[numeric_cols] = yeo.fit_transform(X_train_imputed[numeric_cols])
X_test_imputed[numeric_cols] = yeo.transform(X_test_imputed[numeric_cols])

# Step 5: Min-Max Scaling
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test.columns)

# Step 6: Sampling
def data_sampling(method = "SMOTE-Tomek"):
  if method == "SMOTE-Tomek":
    smt = SMOTETomek(
        sampling_strategy=0.4,
        tomek=TomekLinks(sampling_strategy='all'),
        random_state=42
    )
    X_res, y_res = smt.fit_resample(X_train_scaled, y_train)
    return X_res, y_res

  if method == "ADASYN":
    ada = ADASYN(sampling_strategy='auto', random_state=42)
    X_res, y_res = ada.fit_resample(X_train_scaled, y_train)
    return X_res, y_res

  return X_train_scaled, y_train  # No Sampling

X_res, y_res = data_sampling(
    method = "SMOTE-Tomek"  # ADASYN -- other method
)

# Step 7: Train XGBoost for SHAP
xgb_model = XGBClassifier(tree_method='hist',device='cuda',eval_metric='logloss', random_state=42)
xgb_model.fit(X_res, y_res)

# Step 8: SHAP feature importance
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_res)
shap_importance = np.abs(shap_values.values).mean(axis=0)

shap_feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': shap_importance
}).sort_values(by='importance', ascending=False)

top_features = shap_feature_importance['feature'].head(30).tolist()

# Step 9: Subset to top SHAP features
X_train_shap = X_res[top_features]
X_test_shap = X_test_scaled[top_features]

# Step 10: GridSearchCV on XGBoost
params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1],
    'scale_pos_weight': [1, 3, 5, 10]
}

grid = GridSearchCV(
    XGBClassifier(tree_method='hist',device='cuda',eval_metric='logloss'),
    param_grid=params,
    scoring='f1',
    cv=3,
    n_jobs=-1
)
grid.fit(X_train_shap, y_res)
final_model = grid.best_estimator_
y_pred_xgb = final_model.predict(X_test_shap)

print("🔷 XGBoost GridSearch Results")
print("Best Parameters:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))
print("Top 20 SHAP Features:\n", top_features)

**TabularGAN (synthetic minority oversampling)**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    log_loss, f1_score, average_precision_score, accuracy_score,
    classification_report, confusion_matrix, precision_recall_curve,
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import ADASYN
from imblearn.ensemble import EasyEnsembleClassifier
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
    from sdv.metadata import SingleTableMetadata
    SDV_AVAILABLE = True
except ImportError:
    SDV_AVAILABLE = False

RANDOM_STATE = 42

from google.colab import drive
drive.mount("/content/drive")
TRAIN_PATH = "/content/drive/MyDrive/BTP/train.csv"
TEST_PATH = "/content/drive/MyDrive/BTP/test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
print("Train columns:", list(train_raw.columns))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (67463, 35)
Test shape: (28913, 35)
Train columns: ['ID', 'Loan Amount', 'Funded Amount', 'Funded Amount Investor', 'Term', 'Batch Enrolled', 'Interest Rate', 'Grade', 'Sub Grade', 'Employment Duration', 'Home Ownership', 'Verification Status', 'Payment Plan', 'Loan Title', 'Debit to Income', 'Delinquency - two years', 'Inquires - six months', 'Open Account', 'Public Record', 'Revolving Balance', 'Revolving Utilities', 'Total Accounts', 'Initial List Status', 'Total Received Interest', 'Total Received Late Fee', 'Recoveries', 'Collection Recovery Fee', 'Collection 12 months Medical', 'Application Type', 'Last week Pay', 'Accounts Delinquent', 'Total Collection Amount', 'Total Current Balance', 'Total Revolving Credit Limit', 'Loan Status']


In [ ]:
# Fill missing numeric with median (from train), categorical with mode
def impute_after_clean(X_train, X_other=None):
    X = X_train.copy()
    num_cols = [c for c in NUMERIC_COLS if c in X.columns]
    cat_cols = [c for c in CAT_COLS if c in X.columns]

    for c in num_cols:
        med = X[c].median()
        X[c] = X[c].fillna(med)
        if X_other is not None and c in X_other.columns:
            X_other[c] = X_other[c].fillna(med)
    for c in cat_cols:
        mode_val = X[c].mode().iloc[0] if len(X[c].mode()) > 0 else "Unknown"
        X[c] = X[c].replace("", "Unknown").fillna(mode_val)
        if X_other is not None and c in X_other.columns:
            X_other[c] = X_other[c].replace("", "Unknown").fillna(mode_val)

    return X, X_other

X_full, test_cleaned = impute_after_clean(train_cleaned, test_cleaned)

print("Missing after imputation (train):", X_full.isnull().sum().sum())
print("Missing after imputation (test):", test_cleaned.isnull().sum().sum())

# Drop constant columns (zero variance)
constant_cols = [c for c in X_full.columns if X_full[c].nunique() <= 1]
if constant_cols:
    X_full = X_full.drop(columns=constant_cols)
    test_cleaned = test_cleaned.drop(columns=[c for c in constant_cols if c in test_cleaned.columns])
    print("Dropped constant columns:", constant_cols)

# Align test columns to train
test_cleaned = test_cleaned.reindex(columns=X_full.columns).fillna(0)
print("Final X_full shape:", X_full.shape)
print("Final test shape:", test_cleaned.shape)

# Utilization: Revolving Balance / Total Revolving Credit Limit (avoid div by zero)
if "Total Revolving Credit Limit" in X_full.columns and "Revolving Balance" in X_full.columns:
    X_full["revolving_utilization"] = X_full["Revolving Balance"] / (X_full["Total Revolving Credit Limit"].replace(0, np.nan)).fillna(0)
    test_cleaned["revolving_utilization"] = test_cleaned["Revolving Balance"] / (test_cleaned["Total Revolving Credit Limit"].replace(0, np.nan)).fillna(0)
# Funding gap: how much loan was not funded
if "Loan Amount" in X_full.columns and "Funded Amount" in X_full.columns:
    X_full["funding_gap"] = (X_full["Loan Amount"] - X_full["Funded Amount"]).clip(lower=0)
    test_cleaned["funding_gap"] = (test_cleaned["Loan Amount"] - test_cleaned["Funded Amount"]).clip(lower=0)
# Interest received ratio (signal of repayment behavior)
if "Total Received Interest" in X_full.columns and "Loan Amount" in X_full.columns:
    X_full["interest_ratio"] = X_full["Total Received Interest"] / (X_full["Loan Amount"] + 1)
    test_cleaned["interest_ratio"] = test_cleaned["Total Received Interest"] / (test_cleaned["Loan Amount"] + 1)
# Current balance vs limit (exposure)
if "Total Current Balance" in X_full.columns and "Total Revolving Credit Limit" in X_full.columns:
    X_full["balance_to_limit"] = X_full["Total Current Balance"] / (X_full["Total Revolving Credit Limit"].replace(0, np.nan)).fillna(0)
    test_cleaned["balance_to_limit"] = test_cleaned["Total Current Balance"] / (test_cleaned["Total Revolving Credit Limit"].replace(0, np.nan)).fillna(0)
print("Engineered features added. X_full shape:", X_full.shape)

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape, "X_val:", X_val.shape)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# Yeo-Johnson + MinMax (like Model_Pipeline.ipynb) for better normality
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("yeo", PowerTransformer(method="yeo-johnson", standardize=False)),
    ("scaler", MinMaxScaler()),
])
transformers = [("num", num_pipe, num_cols)]
if cat_cols:
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    transformers.append(("cat", cat_pipe, cat_cols))

preprocessor = ColumnTransformer(transformers, remainder="drop")
preprocessor.fit(X_train)
print("Preprocessor fitted.")

def evaluate(y_true, y_pred, y_proba=None, prefix=""):
    metrics = {}
    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
    metrics["average_precision"] = average_precision_score(y_true, y_proba) if y_proba is not None else 0.0
    if y_proba is not None:
        p = np.clip(y_proba, 1e-15, 1 - 1e-15)
        metrics["log_loss"] = log_loss(y_true, p)
    else:
        metrics["log_loss"] = np.nan
    if prefix:
        print(f"--- {prefix} ---")
    print(f"  Accuracy: {metrics['accuracy']:.4f}  F1: {metrics['f1']:.4f}  PR-AUC: {metrics['average_precision']:.4f}  Log Loss: {metrics.get('log_loss', np.nan):.4f}")
    print(classification_report(y_true, y_pred, target_names=["Non-Defaulter", "Defaulter"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    return metrics

Missing after imputation (train): 0
Missing after imputation (test): 0
Dropped constant columns: ['Payment Plan', 'Accounts Delinquent']
Final X_full shape: (67463, 31)
Final test shape: (28913, 31)
Engineered features added. X_full shape: (67463, 35)
X_train: (53970, 35) X_val: (13493, 35)
Preprocessor fitted.


In [ ]:
TVAE_SYNTHETIC_ROWS = 55_000
MAX_CAT_TVAE = 20

if SDV_AVAILABLE:
    train_table_tvae = X_train.copy()
    train_table_tvae["Loan Status"] = y_train.values
    for col in train_table_tvae.columns:
        if col == "Loan Status":
            continue
        if train_table_tvae[col].dtype == object or train_table_tvae[col].dtype.name == "category":
            counts = train_table_tvae[col].value_counts()
            if len(counts) > MAX_CAT_TVAE:
                top = counts.head(MAX_CAT_TVAE).index.tolist()
                train_table_tvae[col] = train_table_tvae[col].apply(lambda x: x if x in top else "Other")
        else:
            train_table_tvae[col] = pd.to_numeric(train_table_tvae[col], errors="coerce").fillna(0).astype(float)

    metadata_tvae = SingleTableMetadata()
    metadata_tvae.detect_from_dataframe(train_table_tvae)

    print("Fitting TVAE (Tabular VAE) — typically faster than CTGAN...")
    tvae = TVAESynthesizer(metadata_tvae, epochs=150, verbose=False)
    tvae.fit(train_table_tvae)

    print("Sampling synthetic minority rows...")
    synthetic_tvae = tvae.sample(num_rows=TVAE_SYNTHETIC_ROWS + 20000)
    if "Loan Status" in synthetic_tvae.columns:
        synthetic_tvae = synthetic_tvae[synthetic_tvae["Loan Status"] == 1].head(TVAE_SYNTHETIC_ROWS)
    if len(synthetic_tvae) == 0:
        synthetic_tvae = tvae.sample(num_rows=TVAE_SYNTHETIC_ROWS)
        synthetic_tvae["Loan Status"] = 1

    X_syn_tvae = synthetic_tvae.drop(columns=["Loan Status"], errors="ignore")
    for c in X_train.columns:
        if c not in X_syn_tvae.columns:
            X_syn_tvae[c] = 0
    X_syn_tvae = X_syn_tvae[X_train.columns]
    for c in X_syn_tvae.columns:
        if c in X_train.columns and np.issubdtype(X_train[c].dtype, np.number):
            X_syn_tvae[c] = pd.to_numeric(X_syn_tvae[c], errors="coerce").fillna(0)

    X_balanced_tvae = pd.concat([X_train, X_syn_tvae], ignore_index=True)
    y_balanced_tvae = np.concatenate([y_train.values, np.ones(len(X_syn_tvae), dtype=int)])

    X_train_tvae = preprocessor.transform(X_balanced_tvae)
    X_val_tvae = preprocessor.transform(X_val)

    clf_tvae = LGBMClassifier(objective="binary", class_weight="balanced", n_estimators=500, learning_rate=0.1, num_leaves=31, subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0, min_child_weight=10, random_state=RANDOM_STATE, verbose=-1)
    clf_tvae.fit(X_train_tvae, y_balanced_tvae)
    yprob_tvae = clf_tvae.predict_proba(X_val_tvae)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val.values, yprob_tvae)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    best_thresh_tvae = thresholds[np.argmax(f1_scores[:-1])]
    yp_tvae = (yprob_tvae >= best_thresh_tvae).astype(int)
    print(f"TVAE best threshold (by F1): {best_thresh_tvae:.3f}")
    results["TVAE"] = evaluate(y_val.values, yp_tvae, yprob_tvae, prefix="TVAE (validation)")
else:
    print("SDV not installed. Skipping TVAE.")

Fitting TVAE (Tabular VAE) — typically faster than CTGAN...
Sampling synthetic minority rows...
TVAE best threshold (by F1): 0.225
--- TVAE (validation) ---
  Accuracy: 0.1267  F1: 0.1708  PR-AUC: 0.0984  Log Loss: 0.6629
               precision    recall  f1-score   support

Non-Defaulter       0.94      0.04      0.08     12245
    Defaulter       0.09      0.97      0.17      1248

     accuracy                           0.13     13493
    macro avg       0.51      0.51      0.12     13493
 weighted avg       0.86      0.13      0.09     13493

Confusion Matrix:
 [[  495 11750]
 [   34  1214]]


NameError: name 'results' is not defined